# Feature Extraction

In most time-series machine learning tasks, raw signal data is not used directly to train models. Instead, we perform **feature extraction** to transform segments, or "windows", of the time-series into a set of informative features that better represent the underlying patterns. This process converts a sequence of data points into a single row of features that a model can learn from.

**IMPORTANT:** All feature extractor classes in this toolkit are designed to operate on **pre-windowed data**. This means you **must** first use the `Windowing` class to prepare your time-series. The workflow is always:

1.  **Windowing Step:**
2.  **Feature Extraction Step:**

In this notebook, we will demonstrate the three primary feature extraction methods available in the toolkit, following this two-step process.

In [1]:
from ThreeWToolkit.feature_extraction import WindowingConfig, StatisticalConfig, WaveletConfig, EWStatisticalConfig 
from ThreeWToolkit.preprocessing import ImputeMissingConfig, CleanSignalsConfig, SequentialPreprocessingAdapterConfig
from ThreeWToolkit.dataset import ParquetDatasetConfig

## Loading 3W Dataset

Loading the dataset with only real data

In [2]:
# Modify this path to the folder where your dataset is downloaded
dataset_path = "../../dataset"

ds = ParquetDatasetConfig(
    path=dataset_path,
    event_type=["real"],
).build()

print(f"Number of events in the dataset: {len(ds)}")

2026-04-06 16:46:29,700 | INFO | ThreeWToolkit.dataset.parquet_dataset | Dataset found at ../../dataset
2026-04-06 16:46:29,701 | INFO | ThreeWToolkit.dataset.parquet_dataset | Validating dataset integrity...
2026-04-06 16:46:29,703 | INFO | ThreeWToolkit.dataset.parquet_dataset | Dataset integrity check passed!


Number of events in the dataset: 1119


## Custom feature extraction pipelines: 

In order to extract features from events using the ThreeWToolkit, we will import the **TransformConfig** with **SequentialFeatureAdapterConfig** and **ConcatFeatureAdapterConfig** classes, which allow us to create custom pipelines of transformations and apply to a *ParquetDataset* instance. 

In [3]:
from ThreeWToolkit.feature_extraction import SequentialFeatureAdapterConfig, ConcatFeatureAdapterConfig
from ThreeWToolkit.dataset import TransformConfig

Using the **Transformer** class, the *pipeline of feature extraction* can be defined like this: 

In [4]:
transformer = TransformConfig(
    feature_extraction=SequentialFeatureAdapterConfig(
        steps=[
            WindowingConfig(),
            ConcatFeatureAdapterConfig(
                steps=[
                    StatisticalConfig(),
                    EWStatisticalConfig(),
                    WaveletConfig(),
                ]
            ),
        ]
    ),
).build()

To run the feature extraction steps, we will create a default pre processing pipeline to clean the data. 

In [5]:
pre_processing_pipeline = SequentialPreprocessingAdapterConfig(
    steps=[
        CleanSignalsConfig(),
        ImputeMissingConfig(),
    ]
)

Now that we are familiar with the class declarations, let's explore some *feature extraction* methods! 

## Windowing

Function to segment a 1D time-series into overlapping windows and apply a specified windowing function  

The function accepts:  
- A window function (`window`, default = `"hann"`):  
  - A string for standard windows (e.g., `"hann"`, `"hamming"`)  
- A window size (`window_size`, default = 4): number of samples per window  
- An overlap ratio (`overlap`, default = 0.0): must be in the range `[0, 1)`  
- A flag (`pad_last_window`, default = False) to pad the last incomplete window with constant values  
- A padding value (`pad_value`, default = 0.0) used when `pad_last_window=True`  

In [6]:
# Define the pipeline with Windowing followed by Wavelet
data_processor = TransformConfig(
    pre_processing=pre_processing_pipeline,
    feature_extraction=WindowingConfig(window_size=128),
).build()

# Fit the data processor to the dataset (ParquetDataset) and transform the data
data_processor.fit(ds)
transformed_data = data_processor.transform(ds)

In [7]:
single_event = transformed_data[0]

print(f"Shape of the final extracted features: {single_event.signal.shape}")
print("----------------------------------------------")

print(f"Columns of the extracted features: {single_event.signal.columns}")
print("----------------------------------------------")

print(f"First few rows of the extracted features: {single_event.signal.head()}")
print("----------------------------------------------")


Shape of the final extracted features: (2688, 128)
----------------------------------------------
Columns of the extracted features: RangeIndex(start=0, stop=128, step=1)
----------------------------------------------
First few rows of the extracted features:                       0    1    2    3    4    5    6    7    8    9    ...  \
window variable                                                         ...   
0      ESTADO-DHSV    1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  ...   
       ESTADO-M1      1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  ...   
       ESTADO-M2      0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   
       ESTADO-PXO     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   
       ESTADO-SDV-GL  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   

                      118  119  120  121  122  123  124  125  126  127  
window variable                                                         
0      ESTADO-DHSV    1.0  1.0  1.0  1.0

## Wavelet Feature Extraction

The `Wavelet` class uses a signal processing technique called the Stationary Wavelet Transform (SWT). This method decomposes the signal within each window into different frequency components, which can often capture patterns that are invisible to standard statistical measures.

For each level of decomposition, two sets of coefficients are generated:
* **Approximation Coefficients (A):** These capture the low-frequency, underlying trend of the signal. Think of it as a "smoothed" version of the signal within the window.
* **Detail Coefficients (D):** These capture the high-frequency components, representing noise, spikes, and other abrupt changes. 
These features allow a model to differentiate between the general shape of a signal and its more noisy, high-frequency texture.

In [8]:
# Define parameters that will be shared between windowing and feature extraction
# The window_size for the wavelet transform is determined by its level
LEVEL = 7
WINDOW_SIZE = 128
OVERLAP = 0.875

# Define the Windowing config
windowing_config = WindowingConfig(
    window_size=WINDOW_SIZE, overlap=OVERLAP, window="boxcar"
)

In [9]:
# Define the pipeline with Windowing followed by Wavelet
data_processor = TransformConfig(
    pre_processing=pre_processing_pipeline,
    feature_extraction=SequentialFeatureAdapterConfig(
        steps=[
            windowing_config,
            WaveletConfig(level=LEVEL),
        ]
    ),
).build()

# Fit the data processor to the dataset (ParquetDataset) and transform the data
data_processor.fit(ds)
transformed_data = data_processor.transform(ds)

In [10]:
single_event = transformed_data[0]

print(f"Shape of the final extracted features: {single_event.signal.shape}")
print("----------------------------------------------")

print(f"Columns of the extracted features: {single_event.signal.columns}")
print("----------------------------------------------")

print(f"First few rows of the extracted features: {single_event.signal.head()}")
print("----------------------------------------------")


Shape of the final extracted features: (1343, 144)
----------------------------------------------
Columns of the extracted features: Index(['A7_ESTADO-DHSV', 'A7_ESTADO-M1', 'A7_ESTADO-M2', 'A7_ESTADO-PXO',
       'A7_ESTADO-SDV-GL', 'A7_ESTADO-SDV-P', 'A7_ESTADO-W1', 'A7_ESTADO-W2',
       'A7_ESTADO-XO', 'A7_P-ANULAR',
       ...
       'A0_ESTADO-W1', 'A0_ESTADO-W2', 'A0_ESTADO-XO', 'A0_P-ANULAR',
       'A0_P-JUS-CKGL', 'A0_P-MON-CKP', 'A0_P-TPT', 'A0_T-JUS-CKP', 'A0_T-TPT',
       'A0_state'],
      dtype='str', length=144)
----------------------------------------------
First few rows of the extracted features:         A7_ESTADO-DHSV  A7_ESTADO-M1  A7_ESTADO-M2  A7_ESTADO-PXO  \
window                                                              
0            11.313708     11.313708           0.0            0.0   
1            11.313708     11.313708           0.0            0.0   
2            11.313708     11.313708           0.0            0.0   
3            11.313708     11.3

## Statistical Feature Extraction

This is the most common approach to feature extraction for time-series data. The `Statistical` class takes the pre-windowed data and calculates a set of standard statistical descriptors for each window. These features summarize the shape and distribution of the data within that specific time segment.

The features extracted are:
* **`mean`, `std`**: Describe the central tendency and dispersion (volatility).
* **`skew`, `kurtosis`**: Describe the shape of the distribution (asymmetry and presence of outliers).
* **`min`, `1qrt`, `med`, `3qrt`, `max`**: Provide a summary of the distribution through quartiles.

In [11]:
# Define the pipeline with Windowing followed by Wavelet
data_processor = TransformConfig(
    pre_processing=ImputeMissingConfig(),
    feature_extraction=SequentialFeatureAdapterConfig(
        steps=[
            windowing_config,
            StatisticalConfig(features=["mean", "std", "min", "max"]),
        ]
    ),
).build()

# Fit the data processor to the dataset (ParquetDataset) and transform the data
data_processor.fit(ds)
transformed_data = data_processor.transform(ds)

In [12]:
single_event = transformed_data[0]

print(f"Shape of the final extracted features: {single_event.signal.shape}")
print("----------------------------------------------")

print(f"Columns of the extracted features: {single_event.signal.columns}")
print("----------------------------------------------")

print(f"First few rows of the extracted features: {single_event.signal.head()}")
print("----------------------------------------------")

Shape of the final extracted features: (1343, 112)
----------------------------------------------
Columns of the extracted features: Index(['mean_ABER-CKGL', 'mean_ABER-CKP', 'mean_ESTADO-DHSV', 'mean_ESTADO-M1',
       'mean_ESTADO-M2', 'mean_ESTADO-PXO', 'mean_ESTADO-SDV-GL',
       'mean_ESTADO-SDV-P', 'mean_ESTADO-W1', 'mean_ESTADO-W2',
       ...
       'max_P-PDG', 'max_P-TPT', 'max_PT-P', 'max_QBS', 'max_QGL',
       'max_T-JUS-CKP', 'max_T-MON-CKP', 'max_T-PDG', 'max_T-TPT',
       'max_state'],
      dtype='str', length=112)
----------------------------------------------
First few rows of the extracted features:         mean_ABER-CKGL  mean_ABER-CKP  mean_ESTADO-DHSV  mean_ESTADO-M1  \
window                                                                    
0                  0.0            0.0               1.0             1.0   
1                  0.0            0.0               1.0             1.0   
2                  0.0            0.0               1.0             1.0

## Exponentially Weighted Statistical Feature Extraction

The `EWStatistical` class provides a specialized version of the standard statistical features. The "EW" stands for **Exponentially Weighted**.

In this method, not all data points in a window are treated equally. Instead, more recent data points are given progressively higher weight than older points. The rate at which the importance of older data "decays" is controlled by the `decay` parameter.

This is particularly useful in scenarios where the most recent behavior within a window is more predictive of the outcome than the behavior at the beginning of the window. It creates features that are more sensitive to the latest changes in the signal.

In [13]:
# Define the pipeline with Windowing followed by Wavelet
data_processor = TransformConfig(
    pre_processing=pre_processing_pipeline,
    feature_extraction=SequentialFeatureAdapterConfig(
        steps=[
            WindowingConfig(),
            EWStatisticalConfig(window_size=WINDOW_SIZE, 
                                decay=0.9),
        ]
    ),
).build()

# Fit the data processor to the dataset (ParquetDataset) and transform the data
data_processor.fit(ds)
transformed_data = data_processor.transform(ds)

In [14]:
single_event = transformed_data[0]

print(f"Shape of the final extracted features: {single_event.signal.shape}")
print("----------------------------------------------")

print(f"Columns of the extracted features: {single_event.signal.columns}")
print("----------------------------------------------")

print(f"First few rows of the extracted features: {single_event.signal.head()}")
print("----------------------------------------------")

Shape of the final extracted features: (168, 144)
----------------------------------------------
Columns of the extracted features: Index(['ew_mean_ESTADO-DHSV', 'ew_mean_ESTADO-M1', 'ew_mean_ESTADO-M2',
       'ew_mean_ESTADO-PXO', 'ew_mean_ESTADO-SDV-GL', 'ew_mean_ESTADO-SDV-P',
       'ew_mean_ESTADO-W1', 'ew_mean_ESTADO-W2', 'ew_mean_ESTADO-XO',
       'ew_mean_P-ANULAR',
       ...
       'ew_max_ESTADO-W1', 'ew_max_ESTADO-W2', 'ew_max_ESTADO-XO',
       'ew_max_P-ANULAR', 'ew_max_P-JUS-CKGL', 'ew_max_P-MON-CKP',
       'ew_max_P-TPT', 'ew_max_T-JUS-CKP', 'ew_max_T-TPT', 'ew_max_state'],
      dtype='str', length=144)
----------------------------------------------
First few rows of the extracted features:         ew_mean_ESTADO-DHSV  ew_mean_ESTADO-M1  ew_mean_ESTADO-M2  \
window                                                              
0                       1.0                1.0                0.0   
1                       1.0                1.0                0.0   
2   